# Get Variable Data standardized

## Step 0: Import needed metadata

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns 

In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source'],
      dtype='str')

In [4]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

## Step 1: Early experimentation

### Goals
1. Determine the common, exact, names for aggregates of variables.
2. Determine if sub-parts are an issue.
3. Determine if there can be *multiple kinds* of sub-parts.
4. Determine if measurement locations are important.
5. Determine if the multiple kinds of sub-parts in (C) [if any] can be filtered by the measurement locations in (D) [so far, True]

### Option A. Look at all the metrics

Example -- modify as needed

In [5]:
system_id = 1203
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
143,1203,2889,ac_power,AC power,W,W,1.000000,0.000000,ac_meter_1_power_kW+ac_meter_2_power_kW,avg,NaN,NaN,,ac_power__2889
144,1203,2890,dc_power,DC power,W,W,1.000000,0.000000,inv1_dc_power_hW+inv2_dc_power_hW,avg,NaN,NaN,,dc_power__2890
145,1203,2895,inv1_ac_power_hW,AC power,W,W,100.000000,0.000000,,avg,NaN,NaN,,inv1_ac_power_hw__2895
146,1203,2896,inv1_dc_power_hW,DC power,W,W,100.000000,0.000000,,avg,NaN,NaN,,inv1_dc_power_hw__2896
147,1203,2902,inv2_ac_power_hW,AC power,W,W,100.000000,0.000000,,avg,NaN,NaN,,inv2_ac_power_hw__2902
148,1203,2903,inv2_dc_power_hW,DC power,W,W,100.000000,0.000000,,avg,NaN,NaN,,inv2_dc_power_hw__2903
149,1203,2911,ac_inverter_power,AC power,W,W,1.000000,0.000000,inv1_ac_power_hW+inv2_ac_power_hW,avg,NaN,NaN,,ac_inverter_power__2911
150,1203,2898,inv1_ac_current,AC current,A,A,1.000000,0.000000,,avg,NaN,NaN,,inv1_ac_current__2898
151,1203,2905,inv2_ac_current,AC current,A,A,1.000000,0.000000,,avg,NaN,NaN,,inv2_ac_current__2905
152,1203,2909,ac_meter_1_power_kW,AC power,kW,W,1000.000000,0.000000,,avg,NaN,NaN,,ac_meter_1_power_kw__2909


The default example (System 1203) shows that we have inverter ac power (invN_ac_power_hW) and meter ac power (ac_meter_N_power_kW).  This shows that point 3 is nonempty for at least some systems and variables.

Thus, when looking for subsystems, we may need to break it up into pieces, so that we have two kinds of sub-parts.

### Option B: Helper Functions

In [6]:
def metrics_search_for_fragment_df(df: pd.DataFrame, fragment: str):
    '''Search for fragments of a name in sensor_name and common_name'''
    fragment = fragment.lower()
    return df[
        (df.loc[:, 'sensor_name'].str.contains(fragment, case=False))
        | (df.loc[:, 'common_name'].str.contains(fragment, case=False))
    ]

In [7]:
def metrics_search_for_two_fragments_df(df: pd.DataFrame, fragment_1: str,
                                        fragment_2: str, and_or: str):
    '''Search for fragments of two names in sensor_name and common name.
    Use and_or to switch between "both" and "at least one" modes'''
    fragment_1 = fragment_1.lower()
    fragment_2 = fragment_2.lower()
    if and_or == 'and':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            & ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]
    elif and_or == 'or':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            | ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]

Example

In [8]:
# sample use -- search for ac and power
system_id = 1420
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'ac', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1165,1420,4812,ac_power_KwAC,AC power,W,W,1000.0,0.0,,avg,NaN,NaN,AE_HID=37939,ac_power_kwac__4812


Again using System 1203 as the example, note that the aggregate ac_power is the sum of the ac_meter_N_power_hW terms as N varies.  You would not know this from the sensor_name 'ac_power' or the common name 'AC power', though.  Therefore, we sometimes use a wider search, looking at the calc_details and (rarely used) source_type fields.

In [9]:
def widened_search_for_fragment_df(df: pd.DataFrame, fragment: str):
    '''Search for a fragment in calc_details and source_type
    as well as in sensor_name and common_name'''
    fragment = fragment.lower()
    return df[
        (df.loc[:, 'sensor_name'].str.contains(fragment, case=False))
        | (df.loc[:, 'common_name'].str.contains(fragment, case=False))
        | (df.loc[:, 'calc_details'].str.contains(fragment, case=False))
        | (df.loc[:, 'source_type'].str.contains(fragment, case=False))
    ]

In [10]:
def widened_search_for_two_fragments_df(df: pd.DataFrame, fragment_1: str,
                                        fragment_2: str, and_or: str):
    '''Search for two fragments in calc_details and source_type
    as well as in sensor_name and common_name'''
    fragment_1 = fragment_1.lower()
    fragment_2 = fragment_2.lower()
    if and_or == 'and':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'calc_details'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'source_type'].str.contains(fragment_1, case=False)))
            & ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'calc_details'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'source_type'].str.contains(fragment_2, case=False)))
        ]
    elif and_or == 'or':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'calc_details'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'source_type'].str.contains(fragment_1, case=False)))
            | ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'calc_details'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'source_type'].str.contains(fragment_2, case=False)))
        ]

In [11]:
system_id = 1203
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
pow_metrics = metrics_search_for_fragment_df(relevant_rows_metrics, 'pow')
metrics_search_for_two_fragments_df(pow_metrics, 'ac', 'meter', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
152,1203,2909,ac_meter_1_power_kW,AC power,kW,W,1000.0,0.0,,avg,NaN,NaN,,ac_meter_1_power_kw__2909
153,1203,2910,ac_meter_2_power_kW,AC power,kW,W,1000.0,0.0,,avg,NaN,NaN,,ac_meter_2_power_kw__2910


Misses AC power

In [12]:
widened_search_for_two_fragments_df(pow_metrics, 'ac', 'meter', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
143,1203,2889,ac_power,AC power,W,W,1.0,0.0,ac_meter_1_power_kW+ac_meter_2_power_kW,avg,NaN,NaN,,ac_power__2889
152,1203,2909,ac_meter_1_power_kW,AC power,kW,W,1000.0,0.0,,avg,NaN,NaN,,ac_meter_1_power_kw__2909
153,1203,2910,ac_meter_2_power_kW,AC power,kW,W,1000.0,0.0,,avg,NaN,NaN,,ac_meter_2_power_kw__2910


Grabs 'ac_power' appropriately.

## Step 2: Find aggregate power names

### Key variables -- listed for convenience, usually just input to function.

In [24]:
var_name = 'ac_power'
var_short_1 = 'ac'
is_var_short_2 = True
var_short_2 = 'pow'
# need *both* ac and power
and_or = 'and'

#after a manual search, these are the aggregate power names!
ac_agg_sensor_names = ('ac_power', 'ac_power_hW', 'ac_power_kW', 'ac_power_1_6',
                       'InvPAC_kW_Avg', 'ac_power_calc', 'W_avg',
                       'ac_power_metered_kW', 'RTW',
                       'inv_total_ac_power', 'metered_ac_power',
                       'real_power_kW', 'ac_power_KwAC',
                       'ac_power_metered_1_2',
                       'PwrMtrP_kW_Avg')

When would and_or ever be 'or'? Well, when searching for power factor, you either get power_factor or PF, so I search for 'factor' **or** ... 'MtrPF', since 'pf' has false positives with 'tempF'.  Again, the local considerations are legion.

In [14]:
# Feel free to adjust these naming conventions for your variable.
def metadata_agg_name(var_name: str):
    '''Give the name to put in the metadata table.'''
    return f'has_{var_name}_aggregate'


def metadata_agg_subtype_name(var_name: str, source_type: str):
    '''Give the name to put in the metadata table.'''
    return f'has_{var_name}_{source_type}_aggregate'


def find_aggregate_variable_names_gen(var_name: str,
                                      agg_var_sensor_names,
                                      var_short_1: str,
                                      is_var_short_2: bool,
                                      var_short_2: bool,
                                      and_or: str,
                                      print_messages: bool,
                                      sources_matter: bool,
                                      known_sources = ('inverter', 'meter'),
                                      known_sources_short = ('inv', 'met')):
    '''Find all aggregrate variable names for each Parquet system.
    
    Parameters
    -----------
    var_name: str
        The variable name you are searching for
    agg_var_sensor_names: iterable of strings
        The collection of sensor_name entries (exact) for aggregates,
        manually accrued
    var_short_1: str
        The first short string to search for
        For example, 'power' abbreviates to 'pow' in some places, so we search for 'pow'
    is_var_short_2: bool
        if True, look at var_short_2 and and_or
        If False, ignore those values  (but due to Python rules, must put some placeholder there.)
    var_short_2: str
        The second short string to search for.  (Again, must include some placeholder
    and_or: str, "and" or "or"
        If and_or == "and", require both var_short_1 and var_short_2 to be found
        If and_or == "or", require at least one of var_short_1 and var_short_2 to be found.
    print_messages: bool
        If True, prints some message at the end if certain subsystems have no aggregate data,
        or they have too many aggregates.
        If False, does not print these error messages.
        Designed to be set to True while testing, 
        and False when called as a subroutine in later functions.
    sources_matter: bool
        If True, collect data about where the terms are located.
        If False, do not collect such data
    known_sources: iterable of strings
        An iterable of known sources.
    known_sources_short: iterable of strings
        An iterable of shorthands.  Must be the same length as known_sources.
    
    Returns
    -----------
    var_aggs_dict: dict[list[dict]]
        A dictionary, indexed by relevant system_id's.
        The value of var_aggs_dict[system_id] is a list of dictionaries,
        one for each aggregator metric for the systems_id.  Keys for each metric are:
            "metric_id" -- the metric_id number
            "sensor_name" -- the sensor_name term
            "common_name" -- the common-name term
            "units" -- the units for each term
            "whole_or_part" -- determining whether each term is aggregate or a sub-part
            always "whole" for now, but we will add sub-parts in the next function
        If sources_matter = True, then add
            "source_type": the source type if known, or "unknown" if unknown
    var_aggs_metadata: pandas.DataFrame
        If sources_matter = True, then a DataFrame indicating both
        which systems have aggregate variable data, and the aggregate data
        per subtype
        If sources_matter = False, a DataFrame indicating which systems have 
        aggregate variable data only.  
    '''
    # grab metrics_df and systems_cleaned
    # don't literally need this command [not modifying them],
    # but convenient to make dependence explicit
    global metrics_df, systems_cleaned

    # sanitize known_sources and known_sources_short input
    if sources_matter:
        if len(known_sources) != len(known_sources_short):
            raise ValueError('Incorrect match between names and fragments\n'
                             + 'of known_sources and known_sources_short:\n'
                             + f'{len(known_sources)} vs. {len(known_sources_short)}')
        elif 'unknown' not in known_sources:
            # add 'unknown' category to catch all remaining terms
            # and lowercase everything
            known_sources = [
                source_type.lower() for source_type in known_sources
            ]
            known_sources.append('unknown')
            known_sources_short = [
                source_fragment.lower()
                for source_fragment in known_sources_short
            ]
            known_sources_short.append('')  # unknown is a catch-all

    # grab all system_id's of relevant systems
    if is_var_short_2:
        var_data = metrics_search_for_two_fragments_df(
            metrics_df, var_short_1, var_short_2, and_or
        )
    else:
        var_data = metrics_search_for_fragment_df(metrics_df, var_short_1)
    # only reason we need systems_cleaned: 4 "fake" systems
    # with metrics_df system_id but no system_cleaned system_id
    var_system_ids = set(var_data.system_id).intersection(
        set(systems_cleaned.system_id)
    )
    # create receptacles for outer variables
    num_ids = len(var_system_ids)
    var_aggs_dict = {
        system_id: [] for system_id in var_system_ids
    }
    col_names = [metadata_agg_name(var_name)]
    if sources_matter:
        for source_type in known_sources:
            col_names.append(metadata_agg_subtype_name(var_name, source_type))
    num_cols = len(col_names)
    var_aggs_metadata = pd.DataFrame(
        np.full(shape=(num_ids, num_cols), fill_value=False, dtype='bool'),
        index=var_aggs_dict.keys(),
        columns=col_names
    )
    var_aggs_metadata = var_aggs_metadata.sort_index()
    # run through agg_var_sensor_names and see which systems
    # have that exact sensor name
    for sensor_name in agg_var_sensor_names:
        exact_name_metrics =  metrics_df[
                metrics_df['sensor_name'] == sensor_name
        ]
        # again clean against systems_cleaned
        for system_id in set(exact_name_metrics['system_id']).intersection(
            set(systems_cleaned.system_id)
        ):
            relevant_rows_metrics = exact_name_metrics[
                exact_name_metrics['system_id']==system_id
            ]
            if len(relevant_rows_metrics.index) > 1:
                raise RuntimeError(f'System {system_id} has multiple sensors named {sensor_name}!')
            else:
                var_aggs_metadata.loc[system_id, metadata_agg_name(var_name)] = True
                ind = relevant_rows_metrics.index[0]
                metric_id = relevant_rows_metrics.loc[ind, 'metric_id']
                common_name = relevant_rows_metrics.loc[ind, 'common_name']
                given_unit = relevant_rows_metrics.loc[ind, 'units']
                calc_type = relevant_rows_metrics.loc[ind, 'calc_details']
                raw_source_type = relevant_rows_metrics.loc[ind, 'source_type']
                # clean up data -- frequently empty item
                if raw_source_type is np.nan:
                    raw_source_type = 'unknown'
                if sources_matter:
                    for j in range(len(known_sources_short)):
                        source_type = known_sources[j]
                        source_fragment = known_sources_short[j]
                        if (source_fragment in sensor_name.lower())\
                          or (source_fragment in common_name.lower())\
                          or (source_fragment in calc_type.lower())\
                          or (source_fragment in raw_source_type.lower()):
                            var_aggs_dict[system_id].append({
                                'metric_id': metric_id,
                                'sensor_name': sensor_name,
                                'common_name': common_name,
                                'units': given_unit,
                                'calc_details': calc_type,
                                'whole_or_part': 'whole',
                                'source_type': source_type,
                            })
                            # because the last source_fragment is now '',
                            # we get the unknown type for free.
                            var_aggs_metadata.loc[
                                system_id, metadata_agg_subtype_name(var_name, source_type)
                            ] = True
                            # for each particular system_id/sensor_name pair,
                            # only one type, so as soon as a match,
                            break 
                else:
                    # record non-source-type data
                    var_aggs_dict[system_id].append({
                        'metric_id': metric_id,
                        'sensor_name': sensor_name,
                        'common_name': common_name,
                        'units': given_unit,
                        'calc_details': calc_type,
                        'whole_or_part': 'whole',
                    })
    # quick checks for 0 or multiple aggregate values
    if print_messages:
        for system_id in var_system_ids:
            # check for missing entries
            if len(var_aggs_dict[system_id]) == 0:
                print(f'System {system_id} appears to have no obvious {var_name} aggregator name.')
            # check for duplicates
            elif len(var_aggs_dict[system_id]) != 1:
                if sources_matter:
                    # only worry about multiples with the same source_type
                    source_type_counts = {
                        source_type: 0 for source_type in known_sources
                    }
                    for metric_dict in var_aggs_dict[system_id]:
                        source_type_counts[metric_dict['source_type']] += 1
                    for source_type in source_type_counts:
                        if source_type_counts[source_type] > 1:
                            print(f'System {system_id} has multiple {var_name}'
                                  + f'aggregators from the {source_type} source:')
                            for metric_dict in var_aggs_dict[system_id]:
                                print(metric_dict)

                else:  # presumed potentially worrisome
                    print(f'System {system_id} has multiple {var_name} aggregators!')
                    for metric_dict in var_aggs_dict[system_id]:
                        print(metric_dict)

    return (var_aggs_dict, var_aggs_metadata)


Example -- AC Power

In [15]:
var_name = 'ac_power'
var_short_1 = 'ac'
is_var_short_2 = True
var_short_2 = 'pow'
# need *both* ac and power
and_or = 'and'
known_sources = ('inverter', 'meter')
known_sources_short = ('inv', 'met')

#after a manual search, these are the aggregate power names!
ac_agg_sensor_names = ('ac_power', 'ac_power_hW', 'ac_power_kW', 'ac_power_1_6',
                       'InvPAC_kW_Avg', 'ac_power_calc', 'W_avg',
                       'ac_power_metered_kW', 'RTW',
                       'inv_total_ac_power', 'metered_ac_power',
                       'real_power_kW', 'ac_power_KwAC',
                       'ac_power_metered_1_2',
                       'PwrMtrP_kW_Avg')

agg_power_metrics, agg_power_metadata = find_aggregate_variable_names_gen(
    var_name=var_name,
    agg_var_sensor_names=ac_agg_sensor_names,
    var_short_1=var_short_1,
    is_var_short_2=is_var_short_2,
    var_short_2=var_short_2,
    and_or=and_or,
    print_messages=True,
    sources_matter=True
)

System 1200 has multiple ac_poweraggregators from the meter source:
{'metric_id': np.int32(2751), 'sensor_name': 'ac_power', 'common_name': 'AC power', 'units': 'W', 'calc_details': 'ac_power_metered_kW', 'whole_or_part': 'whole', 'source_type': 'meter'}
{'metric_id': np.int32(4197), 'sensor_name': 'ac_power_metered_kW', 'common_name': 'AC power', 'units': 'W', 'calc_details': '', 'whole_or_part': 'whole', 'source_type': 'meter'}
System 1208 has multiple ac_poweraggregators from the meter source:
{'metric_id': np.int32(1016), 'sensor_name': 'ac_power_metered_kW', 'common_name': 'AC power', 'units': 'W', 'calc_details': '', 'whole_or_part': 'whole', 'source_type': 'meter'}
{'metric_id': np.int32(1133), 'sensor_name': 'ac_power_metered_1_2', 'common_name': 'AC power', 'units': 'W', 'calc_details': 'ac_meter_1_power_kW+ac_meter_2_power_kW', 'whole_or_part': 'whole', 'source_type': 'meter'}
System 1283 has multiple ac_poweraggregators from the meter source:
{'metric_id': np.int32(1137), 's

(Comments: For System 1200, ac_power is the power in W and ac_power_metered_kW is the power in kW, despite the misleading units above.
Not sure about System 1208 -- just saw the stray meter here.
For system 1283, ac_power and ac_power_metered_kW seem to be two different things!)

In [16]:
agg_power_metrics

{4: [{'metric_id': np.int32(315),
   'sensor_name': 'ac_power',
   'common_name': 'AC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 10: [{'metric_id': np.int32(423),
   'sensor_name': 'ac_power',
   'common_name': 'AC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 33: [{'metric_id': np.int32(584),
   'sensor_name': 'ac_power',
   'common_name': 'AC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 34: [{'metric_id': np.int32(2695),
   'sensor_name': 'ac_power_hW',
   'common_name': 'AC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 35: [{'metric_id': np.int32(2713),
   'sensor_name': 'ac_power_hW',
   'common_name': 'AC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 36: [{'metric_id'

In [17]:
agg_power_metadata.value_counts()

has_ac_power_aggregate  has_ac_power_inverter_aggregate  has_ac_power_meter_aggregate  has_ac_power_unknown_aggregate
True                    False                            False                         True                              85
                        True                             False                         False                              6
                        False                            True                          False                              6
                        True                             True                          False                              5
Name: count, dtype: int64

### Strategy per variable
1.  Find your list of aggregator names
2.  Determine whether you care about source_type or not.
3.  Run the code.
4.  Amend your list of aggregator names if there are seeming omissions.
5.  Decide if there are special system_id's requiring a more specific code.

### Can I use this verbatim?
Quite possibly!  
AC power doesn't quite work (have to put in the two system-specific issues above), and power-factor doesn't really have or need units, but this has a good chance of working.

## Step 3: Adding on sub-parts

### Helper Function

We isolate our indices by building up a common prefix and suffix.  This is because I do not know whether the names are  `inv_NN_ac_power` or `ac_power_meter_NN` or what.

In [18]:
def common_prefix_and_suffix(names_collection, first_name):
    '''Find the common prefix and suffix of a collection of the strings,
    with the first name in the collection set aside for ease of coding.'''
    common_prefix = ''
    j = 0
    good_prefix = True
    max_len = len(first_name)
    while good_prefix:
        if all(
            [name.startswith(common_prefix) for name in names_collection]
        ):
            j += 1
            common_prefix = first_name[0:j]
            # avoid infinite loop
            if j >= max_len + 1:
                print('Common prefix is whole thing!')
                good_prefix = False
                common_prefix = first_name
        else: # bad prefix, back it up one
            good_prefix = False
            common_prefix = common_prefix[0:-1]
    common_suffix = ''
    j = 0 
    good_suffix = True
    while good_suffix:
        if all(
            [name.endswith(common_suffix) for name in names_collection]
        ):
            j += 1
            common_suffix = first_name[-j:]
            # avoid infinite loop
            if j >= max_len + 1:
                print('Common suffix is whole thing!')
                good_suffix = False
                common_suffix = first_name
        else: # take the last amendment off
            good_suffix = False
            common_suffix = common_suffix[1:]
    return (common_prefix, common_suffix)


In [19]:
# Feel free to adjust these naming conventions for your variable.
def metadata_part_name(var_name: str):
    '''Give the name to put in the metadata table.'''
    return f'has_{var_name}_subsystems'


def metadata_part_subtype_name(var_name: str, source_type: str):
    '''Give the name to put in the metadata table.'''
    return f'has_{var_name}_{source_type}_subsystems'


def find_all_variable_names_gen(var_name: str,
                                agg_var_sensor_names,
                                var_short_1: str,
                                is_var_short_2: bool,
                                var_short_2: bool,
                                and_or: str,
                                print_messages: bool,
                                sources_matter: bool,
                                known_sources = ('inverter', 'meter'),
                                known_sources_short = ('inv', 'met')):
    '''Add subsystem names to aggregation names for each Parquet system.
    
    Parameters
    -----------
    var_name: str
        The variable name you are searching for
    agg_var_sensor_names: iterable of strings
        The collection of sensor_name entries (exact) for aggregates,
        manually accrued
    var_short_1: str
        The first short string to search for
        For example, 'power' abbreviates to 'pow' in some places, so we search for 'pow'
    is_var_short_2: bool
        if True, look at var_short_2 and and_or
        If False, ignore those values  (but due to Python rules, must put some placeholder there.)
    var_short_2: str
        The second short string to search for.  (Again, must include some placeholder
    and_or: str, "and" or "or"
        If and_or == "and", require both var_short_1 and var_short_2 to be found
        If and_or == "or", require at least one of var_short_1 and var_short_2 to be found.
    print_messages: bool
        If True, prints some message at the end if certain subsystems have no aggregate data,
        or they have too many aggregates.
        If False, does not print these error messages.
        Probably should be False here.
    sources_matter: bool
        If True, collect data about where the terms are located.
        If False, do not collect such data
    known_sources: iterable of strings
        An iterable of known sources.
    known_sources_short: iterable of strings
        An iterable of shorthands.  Must be the same length as known_sources.
    
    Returns
    -----------
    var_aggs_dict: dict[list[dict]]
        A dictionary, indexed by relevant system_id's.
        The value of var_aggs_dict[system_id] is a list of dictionaries,
        one for each aggregator metric for the systems_id.  Keys for each metric are:
            "metric_id" -- the metric_id number
            "sensor_name" -- the sensor_name term
            "common_name" -- the common-name term
            "units" -- the units for each term
            "whole_or_part" -- determining whether each term is aggregate or a sub-part
        If sources_matter = True, then add
            "source_type": the source type if known, or "unknown" if unknown
        If a sub-part, add in
            "index" -- the identifying string of the sub-part
    var_aggs_metadata: pandas.DataFrame
        If sources_matter = True, then a DataFrame indicating both
        which systems have aggregate variable data, and the aggregate data
        per subtype
        If sources_matter = False, a DataFrame indicating which systems have 
        aggregate variable data only.  
    '''
    global metrics_df
    # grab part 1 report
    (var_aggs_dict, var_aggs_metadata) = find_aggregate_variable_names_gen(
        var_name=var_name,
        agg_var_sensor_names=agg_var_sensor_names,
        var_short_1=var_short_1,
        is_var_short_2=is_var_short_2,
        var_short_2=var_short_2,
        and_or=and_or,
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    # sanitize known_sources and known_sources_short input
    if sources_matter:
        if len(known_sources) != len(known_sources_short):
            raise ValueError('Incorrect match between names and fragments\n'
                             + 'of known_sources and known_sources_short:\n'
                             + f'{len(known_sources)} vs. {len(known_sources_short)}')
        elif 'unknown' not in known_sources:
            # add 'unknown' category to catch all remaining terms
            # and lowercase everything
            known_sources = [
                source_type.lower() for source_type in known_sources
            ]
            known_sources.append('unknown')
            known_sources_short = [
                source_fragment.lower()
                for source_fragment in known_sources_short
            ]
            known_sources_short.append('')  # unknown is a catch-all
    # prep new terms
    var_total_dict = var_aggs_dict
    num_ids = len(var_aggs_dict.keys())
    col_names = [metadata_part_name(var_name)]
    if sources_matter:
        for source_type in known_sources:
            col_names.append(metadata_part_subtype_name(var_name, source_type))
    num_cols = len(col_names)
    var_parts_metadata = pd.DataFrame(
        np.full(shape=(num_ids, num_cols), fill_value=False, dtype='bool'),
        index=var_aggs_dict.keys(),
        columns=col_names
    )
    # for the last one, we went by sensor_name, and then by system_id
    # here, we work by system_id, and then by name
    for system_id in var_aggs_dict.keys():
        # grab the metrics with the correct system name
        relevant_rows_metrics = metrics_df[metrics_df['system_id']==system_id]
        # grab the metrics that might be important
        if is_var_short_2:
            var_data = metrics_search_for_two_fragments_df(
                relevant_rows_metrics, var_short_1, var_short_2, and_or
            )
        else:
            var_data = metrics_search_for_fragment_df(relevant_rows_metrics, var_short_1)
        # drop the aggregate variable names
        relevant_rows_non_agg_metrics = var_data[
            ~var_data['sensor_name'].isin(agg_var_sensor_names)
        ]
        # is this enough filtering?  That depends on your variable.
        # see if any terms remaining
        num_subparts = relevant_rows_non_agg_metrics.shape[0]
        if num_subparts > 1:  # at least two subparts, keep going!
            var_parts_metadata.loc[system_id, metadata_part_name(var_name)] = True
            if sources_matter:
                for j in range(len(known_sources)):
                    source_type = known_sources[j]
                    source_fragment = known_sources_short[j]
                    var_subparts_known_type = widened_search_for_fragment_df(
                        relevant_rows_non_agg_metrics, source_fragment
                    )
                    num_known_type = var_subparts_known_type.shape[0]
                    if num_known_type > 1:
                        var_parts_metadata.loc[
                            system_id, metadata_part_subtype_name(var_name, source_type)
                        ] = True
                        var_subparts_known_type = var_subparts_known_type.sort_values('sensor_name')
                        var_subparts_known_type_names = var_subparts_known_type['sensor_name'].values
                        first_known_name = var_subparts_known_type_names[0]
                        common_prefix, common_suffix = common_prefix_and_suffix(
                            var_subparts_known_type_names, first_known_name
                        )
                        # add the partial names on there
                        for k in range(0, num_known_type):
                            kth_metric = var_subparts_known_type.iloc[k, :]
                            kth_sensor_name = kth_metric['sensor_name']
                            if kth_sensor_name.startswith(common_prefix)\
                              and kth_sensor_name.endswith(common_suffix):
                                kth_interior = kth_sensor_name.removeprefix(
                                    common_prefix
                                ).removesuffix(common_suffix)
                            else:
                                raise ValueError('Bad prefix or suffix!') 
                            var_total_dict[system_id].append({
                                'metric_id': kth_metric['metric_id'],
                                'sensor_name': kth_sensor_name,
                                'common_name': kth_metric['common_name'],
                                'units': kth_metric['units'],
                                'calc_details': kth_metric['calc_details'],
                                'whole_or_part': 'part',
                                'source_type': source_type,
                                'index': kth_interior
                            })
                        # avoid repeats by dropping the used terms
                        relevant_rows_non_agg_metrics = relevant_rows_non_agg_metrics.drop(
                            index = var_subparts_known_type.index
                        )
                    elif num_known_type == 1: # only one sub-part?  Something's wrong.
                        print(f'System {system_id} has only one {source_type}-type '
                              + f'subpart for {var_name}!')
                        print(var_subparts_known_type.iloc[0, :])
                        raise ValueError('Incorrect subpart description, presumably.')
                    # if 0 of known type, just pass on!
            else: # no source_type, just start making prefixes and suffixes
                relevant_rows_non_agg_metrics = relevant_rows_non_agg_metrics.sort_values('sensor_name')
                relevant_rows_non_agg_names = relevant_rows_non_agg_metrics['sensor_name'].values
                first_name = relevant_rows_non_agg_names[0]
                (common_prefix, common_suffix) = common_prefix_and_suffix(
                    relevant_rows_non_agg_names, first_name
                )
                for k in range(0, num_subparts):
                    kth_metric = relevant_rows_non_agg_metrics.iloc[k, :]
                    kth_sensor_name = kth_metric['sensor_name']
                    if kth_sensor_name.startswith(common_prefix)\
                      and kth_sensor_name.endswith(common_suffix):
                        kth_interior = kth_sensor_name.removeprefix(common_prefix).removesuffix(common_suffix)
                    else:
                        raise ValueError('Bad prefix or suffix!') 
                    var_total_dict[system_id].append({
                        'metric_id': kth_metric['metric_id'],
                        'sensor_name': kth_sensor_name,
                        'common_name': kth_metric['common_name'],
                        'units': kth_metric['units'],
                        'calc_details': kth_metric['calc_details'],
                        'whole_or_part': 'part',
                        'index': kth_interior
                        })
        elif num_subparts == 1:  # only one subpart?  Presumably a missing aggregate name
            print(f'System {system_id} has only one subpart for {var_name}!')
            print(relevant_rows_non_agg_metrics.iloc[0, :])
            raise ValueError('Incorrect subpart description, presumably.')
        # if 0 sub-parts, move to the next system_id
    var_total_metadata_df = pd.merge(left=var_aggs_metadata,
                                     right=var_parts_metadata,
                                     how='inner',
                                     left_index=True,
                                     right_index=True)
    return (var_total_dict, var_total_metadata_df)


#### Nice example: power_factor

In [20]:
var_name = 'power_factor'
var_short_1 = 'factor'
is_var_short_2 = True
var_short_2 = 'MtrPF'
# need *one* of 'factor', 'MtrPF'
and_or = 'or'
known_sources = ('inverter', 'meter')
known_sources_short = ('inv', 'met')

#after a manual search, these are the aggregate power names!
pf_sensor_names = ('power_factor', 'power_factor_1000',
                       'PwrMtrPF_Avg')

all_power_factor_metrics, all_power_factor_metadata = find_all_variable_names_gen(
    var_name=var_name,
    agg_var_sensor_names=pf_sensor_names,
    var_short_1=var_short_1,
    is_var_short_2=is_var_short_2,
    var_short_2=var_short_2,
    and_or=and_or,
    print_messages=True,
    sources_matter=True,
    known_sources=known_sources,
    known_sources_short=known_sources_short
)

System 1203 appears to have no obvious power_factor aggregator name.
System 1278 appears to have no obvious power_factor aggregator name.


In [21]:
all_power_factor_metrics

{33: [{'metric_id': np.int32(596),
   'sensor_name': 'power_factor',
   'common_name': 'AC other',
   'units': '-',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 34: [{'metric_id': np.int32(2693),
   'sensor_name': 'power_factor_1000',
   'common_name': 'AC other',
   'units': '-',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 35: [{'metric_id': np.int32(2711),
   'sensor_name': 'power_factor_1000',
   'common_name': 'AC other',
   'units': '-',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 1284: [{'metric_id': np.int32(957),
   'sensor_name': 'power_factor',
   'common_name': 'AC other',
   'units': '-',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 4901: [{'metric_id': np.int32(82542),
   'sensor_name': 'PwrMtrPF_Avg',
   'common_name': 'AC power other',
   'units': '-',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_t

In [22]:
all_power_factor_metadata.value_counts()

has_power_factor_aggregate  has_power_factor_inverter_aggregate  has_power_factor_meter_aggregate  has_power_factor_unknown_aggregate  has_power_factor_subsystems  has_power_factor_inverter_subsystems  has_power_factor_meter_subsystems  has_power_factor_unknown_subsystems
True                        False                                False                             True                                False                        False                                 False                              False                                  11
                                                                 True                              False                               False                        False                                 False                              False                                   3
False                       False                                False                             False                               True                         True 

#### Needed sanity check!

See if the "parts" are actual parts, or just aggregate names you missed.

In [25]:
part_metric_names = []
for system_id in all_power_factor_metrics.keys():
    for metric_dict in all_power_factor_metrics[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

inv1_power_factor
inv1_power_factor
inv2_power_factor
inv2_power_factor
power_factor_phA
power_factor_phB
power_factor_phC


(Checking to make sure that all units are valid is also a good idea.)

#### Bad example: ac power

In [121]:
var_name = 'ac_power'
var_short_1 = 'ac'
is_var_short_2 = True
var_short_2 = 'pow'
# need *both* ac and power
and_or = 'and'
known_sources = ('inverter', 'meter')
known_sources_short = ('inv', 'met')

#after a manual search, these are the aggregate power names!
ac_agg_sensor_names = ('ac_power', 'ac_power_hW', 'ac_power_kW', 'ac_power_1_6',
                       'InvPAC_kW_Avg', 'ac_power_calc', 'W_avg',
                       'ac_power_metered_kW', 'RTW',
                       'inv_total_ac_power', 'metered_ac_power',
                       'real_power_kW', 'ac_power_KwAC',
                       'ac_power_metered_1_2',
                       'PwrMtrP_kW_Avg')

all_power_metrics, all_power_metadata = find_all_variable_names_gen(
    var_name=var_name,
    agg_var_sensor_names=ac_agg_sensor_names,
    var_short_1=var_short_1,
    is_var_short_2=is_var_short_2,
    var_short_2=var_short_2,
    and_or=and_or,
    print_messages=False,
    sources_matter=True
)

System 4 has only one subpart for ac_power!
system_id                           4
metric_id                         327
sensor_name              power_factor
common_name                  AC other
raw_units                          na
units                              na
calc_scale                        1.0
calc_offset                       0.0
calc_details                         
aggregation_type                  avg
source_type                       NaN
source_id                         NaN
comments                             
standard_name       power_factor__327
Name: 1714, dtype: object


ValueError: Incorrect subpart description, presumably.

Trouble -- grabs all the power-factor, energy, complex-power terms too!

### Can I use this verbatim?
This is more doubtful, as there are often some system-specific issues.  Examples:

**DC Power**: For System 1207, the aggregate power has watts, but the individual powers have no units.

Also, for System 1416, there is a dc_power_positive and dc_power_negative, which do not seem to be subsystems in the same way, so I manually discarded them.

**AC Power**: As above, have to filter out the power-factor terms, etc.  [It suffices here to limit units to watts and kilowatts, as the System-1207 issue does not arise and everything else has strange units.]

### Strategy:
1.  Try using the code above.
2.  If there are errors, you may need to make a variable-specific variant to do better filtering, etc.
3.  Even if there are no errors, you should carefully inspect the output and see if there are any strange variables that are not assigned appropriately.  If this is a problem (and it is not solved by adjusting the inputs), custom code may again be necessary.
4.  If you do need to make a custom code, go ahead and hardcode the aggregate-value sets if you can.  Also, once you are comfortable with the variable, you can adjust to the with-source-type or without-source-type variant as necessary.

## Step 4: Make DataFrame with data, per-system

Naming functions for power factors

In [ ]:
# can and should adjust these naming functions
# in particular, if there are a lot of unknown-source-type systems,
# may want to omit source_type if unknown
def var_agg_name(var_name: str, source_type: str, has_subparts: bool, agg_type: str):
    '''Make the standardized variable name for the aggregate value.
    
    Parameters
    -----------
    var_name: str
        The name of the variable we are studying
    source_type: str or None
        If not None, the source_type of the variable
    has_subparts: bool
        Whether or not there are any subparts.
    agg_type: str, 'sum' or 'mean'
        How the sub-parts are combined.
        Only used if has_subparts = True
        For most variables, agg_type is 'sum',
        but for temperature, it is almost certainly averaged.
    '''
    agg_name = var_name
    if source_type is not None:  # and source_type != 'unknown' ??
        agg_name = agg_name + '_' + source_type
    if has_subparts:
        agg_name = agg_name + '_' + agg_type
    return agg_name


def var_part_name(var_name: str, source_type: str, ind):
    '''Make the standardized variable name for the subpart.
    
    Parameters
    -----------
    var_name: str
        The name of the variable we are studying
    source_type: str or None
        If not None, the source_type of the variable
    ind: int or str
        The index of the term.  Can be an int or a string. 
    '''
    if source_type is not None:
        return f'{var_name}_{source_type}_{ind}'
    else:
        return f'{var_name}_{ind}'

In [150]:
def var_dataframe_generator(system_id: int, 
                            tall_or_wide: str,
                            error_on_no_data: bool,
                            order_priority: str,
                            agg_type: str,
                            add_aggs: bool,
                            var_name: str,
                            agg_var_sensor_names,
                            var_short_1: str,
                            is_var_short_2: bool,
                            var_short_2: bool,
                            and_or: str,
                            print_messages: bool,
                            sources_matter: bool,
                            known_sources = ('inverter', 'meter'),
                            known_sources_short = ('inv', 'met')):
    '''Make the (tall or wide) pandas DataFrame with all variable data
    from a Parquet system.
    
    Parameters
    ----------
    system_id: int
        Index of system in systems_cleaned and metric_df
    tall_or_wide: str
        If 'wide', return wide Dataframe
        if 'tall', convert back to a 3-column array.
    error_in_no_data: bool
        If True, return an error if the system_id has no power-factor data.
        If False, return None if the system-system_id has no power factor data.
    order_priority: str, "whole_before_part" or "connect_like_terms"
        If "whole_before_part", puts all aggregate figures before all subdata_figures
        If "connect_like_terms", lists inverter aggregate, then inverter parts,
            then meter aggregate, then meter parts, then unknown together, then unknown parts.
            Has no effect if sources_matter = False
    agg_type: str, 'sum' or 'mean'
        How are the sub-parts being combined?
        For most variables, you just sum across sub-units,
        But temperature, and possibly power_factor, should be averaged.
    add_aggs: bool
        If True, and there are parts without a corresponding aggregate,
            add the aggregate, according to agg_type.
        If False, do nothing.
    var_name: str
        The variable name you are searching for
    agg_var_sensor_names: iterable of strings
        The collection of sensor_name entries (exact) for aggregates,
        manually accrued
    var_short_1: str
        The first short string to search for
        For example, 'power' abbreviates to 'pow' in some places, so we search for 'pow'
    is_var_short_2: bool
        if True, look at var_short_2 and and_or
        If False, ignore those values  (but due to Python rules, must put some placeholder there.)
    var_short_2: str
        The second short string to search for.  (Again, must include some placeholder
    and_or: str, "and" or "or"
        If and_or == "and", require both var_short_1 and var_short_2 to be found
        If and_or == "or", require at least one of var_short_1 and var_short_2 to be found.
    print_messages: bool
        If True, prints some message at the end if certain subsystems have no aggregate data,
        or they have too many aggregates.
        If False, does not print these error messages.
        Probably should be False here.
    sources_matter: bool
        If True, collect data about where the data is collected (meter, inverter, etc.),
            and also stratify data by source.
        If False, do not collect such data.
    known_sources: iterable of strings
        An iterable of known sources.
    known_sources_short: iterable of strings
        An iterable of shorthands.  Must be the same length as known_sources.

    Returns
    ---------
    A pandas DataFrame object with the desired data.
    '''
    # Grab the variable_names organized earlier
    all_var_metrics, all_var_metadata = find_all_variable_names_gen(
        var_name=var_name,
        agg_var_sensor_names=agg_var_sensor_names,
        var_short_1=var_short_1,
        is_var_short_2=is_var_short_2,
        var_short_2=var_short_2,
        and_or=and_or,
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    try:
        my_var_metrics = all_var_metrics[system_id]
        my_metadata = all_var_metadata.loc[system_id, :]
    except KeyError:
        if error_on_no_data:
            raise ValueError(f'System {system_id} has no {var_name} data!')
        else:
            return None
    except BaseException as e:
        raise e
    # sanitize known_sources and known_sources_short input
    if sources_matter:
        if len(known_sources) != len(known_sources_short):
            raise ValueError('Incorrect match between names and fragments\n'
                             + 'of known_sources and known_sources_short:\n'
                             + f'{len(known_sources)} vs. {len(known_sources_short)}')
        elif 'unknown' not in known_sources:
            # add 'unknown' category to catch all remaining terms
            # and lowercase everything
            known_sources = [
                source_type.lower() for source_type in known_sources
            ]
            known_sources.append('unknown')
            known_sources_short = [
                source_fragment.lower()
                for source_fragment in known_sources_short
            ]
            known_sources_short.append('')  # unknown is a catch-all
    # some quick reads of the data
    metric_ids = []
    whole_metric_ids = []
    if sources_matter:
        source_type_metric_ids = {
            source_type: [] for source_type in known_sources
        }
    # grab all metric ids, putting the 'whole' category first
    for metric_data_dict in my_var_metrics:
        # whole-part distribution
        if metric_data_dict['whole_or_part'] == 'whole':
            metric_ids.insert(0, metric_data_dict['metric_id'])
            whole_metric_ids.append(metric_data_dict['metric_id'])
        elif metric_data_dict['whole_or_part'] == 'part':
            metric_ids.append(metric_data_dict['metric_id'])
        else:
            raise ValueError('The "whole_or_part" result of find_all_variable_names_gen()\n'
                             f'is not correct for system {system_id}.')
        # get source-type metric updated.
        if sources_matter:
            source_type_metric_ids[metric_data_dict['source_type']].append(
                metric_data_dict['metric_id']
            )
    # Load only these metrics from the system
    my_system_parquet_data_path = Path(f'../../../../data_ds_project/systems/parquet/{system_id}/')
    my_system_parquet_selection = pq.ParquetDataset(
        my_system_parquet_data_path, filters=[
            ('metric_id', 'in', metric_ids)
        ]
    )
    system_df = my_system_parquet_selection.read().to_pandas()
    # for reference, 4 columns (see
    # https://github.com/openEDI/documentation/blob/main/pvdaq.md#pvdaq_pvdata)
    # measured_on, utc_measured_on, metric_id, value
    # standard cleaning
    system_df = system_df.drop_duplicates()
    # See if multiple values at a given time
    # if so, forced to replace value by mean value
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id'])):
        system_df.loc[:, 'mean_value'] = system_df.groupby(
            ['measured_on', 'metric_id']
        )['value'].transform('mean')
        system_df = system_df.drop(columns='value')
        system_df = system_df.rename(columns={'mean_value':'value'})
        system_df.drop_duplicates()
    # if still duplicates, forced to drop utc_measured_on,
    # a frequent source of off-by-one-hour errors
    # (and points with the same 'measured_on' but different 'utc_measured_on'
    # have the same value, so it is likely that utc_measured_on is the problem)
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id', 'value'])):
        system_df = system_df.drop(columns='utc_measured_on')
        system_df = system_df.drop_duplicates()
    # ready to widen the columns
    wide_df = system_df.pivot(
        index='measured_on',
        columns='metric_id',
        values='value'
    )
    # reset the metric_id name of the index of columns
    wide_df.columns.name = ''
    # reset the index
    wide_df = wide_df.reset_index()
    # Some systems have part-data and not aggregate data;  
    # amend this mistake.
    if add_aggs:
        if sources_matter:
            for source_type in known_sources:
                if (my_metadata[metadata_part_subtype_name(var_name, source_type)])\
                  and (not my_metadata[metadata_agg_subtype_name(var_name, source_type)]):
                    source_type_total_name = var_agg_name(
                        var_name, source_type, True, agg_type
                    )
                    if agg_type == 'sum':
                        wide_df.loc[:, source_type_total_name] = wide_df.apply(
                            lambda row: np.sum(
                                [row[j] for j in source_type_metric_ids[source_type]]
                            ), axis=1
                        )
                    elif agg_type == 'mean':
                        wide_df.loc[:, source_type_total_name] = wide_df.apply(
                            lambda row: np.mean(
                                [row[j] for j in source_type_metric_ids[source_type]]
                            ), axis=1
                        )
                    else:
                        raise ValueError(f'Bad agg_type = {agg_type}.'
                                         + ' Should be "sum" or "mean".')
                    whole_metric_ids.append(source_type_total_name)
                    source_type_metric_ids[source_type].append(
                        source_type_total_name
                    )
        elif my_metadata[metadata_part_name(var_name)]\
          and (not my_metadata[metadata_agg_name(var_name)]):
            total_name = var_agg_name(var_name, None, True, agg_type)
            if agg_type == 'sum':
                wide_df.loc[:, total_name] = wide_df.apply(
                    lambda row: np.sum(
                        [row[j] for j in wide_df.columns[1:]]
                    ), axis=1
                )
            elif agg_type == 'mean':
                wide_df.loc[:, total_name] = wide_df.apply(
                    lambda row: np.mean(
                        [row[j] for j in wide_df.columns[1:]]
                    ), axis=1
                )
            else:
                raise ValueError(f'Bad agg_type = {agg_type}.'
                                 + ' Should be sum or mean.')
            whole_metric_ids.append(total_name)
    # reorder columns according to order_priority
    if order_priority == 'whole_before_part':
        # push the 'whole' columns to the beginning of the pack
        # despite re-ordering earlier, can still be loaded in the wrong order.
        reordered_columns = ['measured_on'] + whole_metric_ids + (wide_df.columns.drop(
            ['measured_on'] + whole_metric_ids
        ).tolist())
        wide_df = wide_df[reordered_columns]
    elif (order_priority == 'connect_like_terms') and sources_matter:
        reordered_columns = ['measured_on',]
        for source_type in known_sources:
            cols_known_type = []
            for j in source_type_metric_ids[source_type]:
                if j in whole_metric_ids:
                    cols_known_type.insert(0, j)
                else:
                    cols_known_type.append(j)
            reordered_columns = reordered_columns + cols_known_type
        wide_df = wide_df[reordered_columns]
    # rename columns
    renamer_dict = dict()
    if sources_matter:
        for metric_data_dict in my_var_metrics:
            source_type = metric_data_dict['source_type']
            if metric_data_dict['whole_or_part'] == 'whole':
                renamer_dict[metric_data_dict['metric_id']] = var_agg_name(
                    var_name,
                    source_type,
                    my_metadata[
                        metadata_agg_subtype_name(var_name, source_type)
                    ], 
                    agg_type
                )
            elif metric_data_dict['whole_or_part'] == 'part':
                renamer_dict[metric_data_dict['metric_id']] = var_part_name(
                    var_name,
                    source_type,
                    metric_data_dict['index']
            )
            else:
                raise ValueError('The "whole_or_part" result of '
                                 + 'find_all_variable_names_gen()\n'
                                 f'is not correct for system {system_id}.')
    else:
        for metric_data_dict in my_var_metrics:
            if metric_data_dict['whole_or_part'] == 'whole':
                renamer_dict[metric_data_dict['metric_id']] = var_agg_name(
                    var_name,
                    None,
                    my_metadata[metadata_part_name(var_name)],
                    agg_type
                )
            elif metric_data_dict['whole_or_part'] == 'part':
                renamer_dict[metric_data_dict['metric_id']] = var_part_name(
                    var_name,
                    None,
                    metric_data_dict['index']
                )
            else:
                raise ValueError('The "whole_or_part" result of '
                                 + 'find_all_variable_names_gen()\n'
                                 f'is not correct for system {system_id}.')
    wide_df = wide_df.rename(columns=renamer_dict)
    # convert back to tall format if that is what we wanted
    if tall_or_wide == 'wide':
        return wide_df
    elif tall_or_wide == 'tall':
        return wide_df.melt(
            id_vars='measured_on',
            value_vars=wide_df.columns[1:],
            var_name='metric_name',
            value_name='value'
        )
    else:
        raise ValueError('The term "tall_or_wide" must be "tall" or "wide.\n'
                         + f'Recieved {tall_or_wide}')


### Test -- power-factor

In [147]:
var_name = 'power_factor'
var_short_1 = 'factor'
is_var_short_2 = True
var_short_2 = 'MtrPF'
# need *one* of 'factor', 'MtrPF'
and_or = 'or'
known_sources = ('inverter', 'meter')
known_sources_short = ('inv', 'met')

#after a manual search, these are the aggregate power names!
pf_sensor_names = ('power_factor', 'power_factor_1000',
                       'PwrMtrPF_Avg')

var_dataframe_generator(
    system_id = 1203,  # Try varying this!
    tall_or_wide='wide',  # Try varying this!
    error_on_no_data=False,
    order_priority = 'connect_like_terms',  # Try varying this!
    agg_type='mean',
    add_aggs=True,  # Try varying this!
    var_name=var_name,
    agg_var_sensor_names=pf_sensor_names,
    var_short_1=var_short_1,
    is_var_short_2=is_var_short_2,
    var_short_2=var_short_2,
    and_or=and_or,
    print_messages=False,
    sources_matter=True,  # Try varying this!
    known_sources=known_sources,
    known_sources_short=known_sources_short
)

,measured_on,power_factor_inverter_mean,power_factor_inverter_1,power_factor_inverter_2
0,2011-01-19 14:30:01,997.0,999.0,995.0
1,2011-01-19 14:45:01,987.5,981.0,994.0
2,2011-01-19 14:48:02,985.5,978.0,993.0
3,2011-01-19 14:51:02,977.5,976.0,979.0
4,2011-01-19 14:54:01,991.5,989.0,994.0
...,...,...,...,...
1545262,2020-07-26 16:55:02,NaN,NaN,999.0
1545263,2020-07-26 17:00:01,NaN,1000.0,NaN
1545264,2020-07-26 17:00:02,NaN,NaN,999.0
1545265,2020-07-26 17:05:01,NaN,999.0,NaN


### Can I use it verbatim?

Doubtful.  In particular, if you had to modify Step 3 (grabbing all variable names and metadata), it would show up again here.

Also, if it does not make sense to create an aggregator when it doesn't exist, you can drop the add_aggs and aggs_type inputs, modify the variable names accordingly, and presto, a savings of 40 lines!